### Imports

In [38]:
import os
from dotenv import load_dotenv
import fitz  # PyMuPDF
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA
from langchain_core.documents import Document

print("All imports successful!")

All imports successful!


OpenRouter config

In [39]:
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

# LLM — OpenRouter through OpenAI-compatible API
llm = ChatOpenAI(
    model="deepseek/deepseek-chat-v3-0324",
    openai_api_key=api_key,
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.3,
)


from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("LLM + Embeddings ready!")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

LLM + Embeddings ready!


PDF Loader

In [40]:

def load_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    documents = []
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text()
        
        if text.strip():  
            documents.append(Document(
                page_content=text,
                metadata={
                    "page": page_num + 1,
                    "source": pdf_path
                }
            ))
    
    print(f"Total pages loaded: {len(documents)}")
    return documents

Text Splitter

In [41]:
def split_documents(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100,
    )
    
    chunks = splitter.split_documents(documents)
    print(f"Total chunks created: {len(chunks)}")
    return chunks

Vector Store (FAISS)

In [42]:
 
def create_vector_store(chunks):
    print("Creating embeddings...")
    
    vector_store = FAISS.from_documents(chunks, embeddings)
    vector_store.save_local("../data/faiss_index")
    
    print("Vector store saved!")
    return vector_store

RAG Chain

In [45]:

def create_rag_chain(vector_store):
    retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 2,
        "fetch_k": 10
    }
    )
    
    chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True,
    )
    
    print("RAG chain ready!")
    return chain

Test

In [46]:
pdf_path = "../data/your_file.pdf"

docs     = load_pdf(pdf_path)
chunks   = split_documents(docs)
vs       = create_vector_store(chunks)
chain    = create_rag_chain(vs)

question = "Which machine learning type uses rewards and penalties?"
result   = chain.invoke({"query": question})

print("\nAnswer:\n", result["result"])
print("\nSources:")
for doc in result["source_documents"]:
    print("="*50)
    print("Page:", doc.metadata["page"])
    print(doc.page_content)

Total pages loaded: 3
Total chunks created: 4
Creating embeddings...
Vector store saved!
RAG chain ready!

Answer:
 The machine learning type that uses rewards and penalties is **Reinforcement Learning**. 

In reinforcement learning, an agent learns by interacting with an environment, receiving rewards for desirable actions and penalties (or negative rewards) for undesirable ones. Examples include game-playing AI and robotics. 

Let me know if you'd like further clarification!

Sources:
Page: 1
2. Unsupervised Learning - Uses unlabeled data. Examples: Customer segmentation, clustering.
3. Reinforcement Learning - Learns through rewards and penalties. Examples: Game-playing AI,
robotics.
Machine Learning is widely used in healthcare, finance, cybersecurity, and recommendation
systems.
Page: 3
Generative AI and Large Language Models
Generative AI focuses on creating new content such as text, images, audio, and code.
Large Language Models (LLMs) are trained on massive datasets and can gen